In [ ]:
# This notebook is designed and expected to run in the environment of Google Colab. The dependency on the gemini automated commentary further enforces this decision.

# Installation - The packages listed below are typically pre-installed in Google Colab environments, so their re-installation is redundant.
# !pip install requests pandas numpy matplotlib scikit-learn fpdf

# A Coingecko API KEY is required to run this notebook. Obtaining one is free and achievable in less than 2 minutes.
# 1. Get a demo API key for free after sign-up, on: https://www.coingecko.com/en/api
# 2. Paste the key below, between the ""
# 3. Run everything!

# Important: Avoid Spamming 'Run All'
# CoinGecko's and DefiLlama's public APIs have rate limits to prevent abuse. Running the notebook multiple times in quick succession (e.g., via 'Runtime > Run all') may trigger timeouts or temporary blocks,
# causing fetches to fail. Wait 1-2 minutes between runs if testing, or run cells incrementally. If you hit a 429 error, pause for 5-10 minutes before retrying.

API_KEY = " "

In [ ]:
!pip install fpdf

import requests
import pandas as pd
import numpy as np
import time
import re
import os
import matplotlib.pyplot as plt
from fpdf import FPDF
from datetime import datetime
from google.colab import ai

OUTPUT_FOLDER = "defi_report_assets"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [ ]:
# CONFIG

PROTOCOLS = {
    "Pendle": {
        "coingecko_id": "pendle",
        "defillama_slug": "pendle",
        "revenue_accrues_to_holder": True     # vePENDLE lockers receive protocol fees
    },
    "Ethena": {
        "coingecko_id": "ethena",
        "defillama_slug": "ethena",
        "revenue_accrues_to_holder": False    # yield accrues to sUSDe stakers, not ENA holders
    },
    "GMX": {
        "coingecko_id": "gmx",
        "defillama_slug": "gmx",
        "revenue_accrues_to_holder": True     # GMX stakers receive esGMX + ETH fee distributions
    },
    "Lido": {
        "coingecko_id": "lido-dao",
        "defillama_slug": "lido",
        "revenue_accrues_to_holder": False    # revenue flows to DAO treasury - LDO is governance only
    },
    "Curve Finance": {
        "coingecko_id": "curve-dao-token",
        "defillama_slug": "curve-finance",
        "revenue_accrues_to_holder": True     # veCRV lockers receive 50% of trading fees + bribes
    },
    "Sky": {
        "coingecko_id": "sky",
        "defillama_slug": "sky",
        "revenue_accrues_to_holder": True     # SKY stakers receive DSR spread and RWA yield
    }
}


COINGECKO_CACHE = {}                           # Global cache so we never call the same coin twice

COLORS = {
    "Lido": "#00A3FF",
    "Pendle": "#FF6B00",
    "Curve Finance": "#FF5C5C",
    "GMX": "#4C6FFF",
    "Ethena": "#0A8F8C",
    "Sky": "#1A73E8"
}



# fix for fpdf incompatible characters by stripping or replacing Unicode characters that fpdf cannot encode in latin-1
def sanitize(text):
    replacements = {
        "\u2018": "'", "\u2019": "'",   # smart single quotes
        "\u201c": '"', "\u201d": '"',   # smart double quotes
        "\u2013": "-", "\u2014": "-",   # en-dash, em-dash
        "\u2026": "...",                # ellipsis
        "\u00b7": "-",                  # middle dot
        "\u2022": "-",                  # bullet
    }
    for char, replacement in replacements.items():
        text = text.replace(char, replacement)
    # Final fallback: drop anything still outside latin-1 range
    return text.encode("latin-1", errors="ignore").decode("latin-1")

In [ ]:
# DATA FETCHING

def fetch_coingecko_data(coin_id, api_key=None):
    if coin_id in COINGECKO_CACHE:
        print(f"CoinGecko {coin_id} from cache")
        return COINGECKO_CACHE[coin_id]

    url = f"https://api.coingecko.com/api/v3/coins/{coin_id}"
    headers = {}
    if api_key:
        headers["x-cg-demo-api-key"] = api_key

    for attempt in range(5):
        r = requests.get(url, headers=headers)
        print(f"CoinGecko {coin_id} → Status: {r.status_code}")

        if r.status_code == 200:
            data = r.json()
            result = {
                "price": data["market_data"]["current_price"]["usd"],
                "market_cap": data["market_data"]["market_cap"]["usd"],
                "circulating_supply": data["market_data"]["circulating_supply"],
                "total_supply": data["market_data"]["total_supply"],
                "max_supply": data["market_data"]["max_supply"]
            }
            COINGECKO_CACHE[coin_id] = result
            return result

        elif r.status_code == 429:
            wait = (2 ** attempt) * 5
            print(f"⏳ Rate limit hit. Waiting {wait}s...")
            time.sleep(wait)
        else:
            print(f"CoinGecko error {r.status_code}: {r.text[:200]}")
            time.sleep(2 ** attempt)

    raise Exception(f"Failed to fetch CoinGecko data for {coin_id} after retries")



def fetch_tvl_history(protocol_slug):
    url = f"https://api.llama.fi/protocol/{protocol_slug}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36"
    }

    for attempt in range(3):
        try:
            r = requests.get(url, headers=headers, timeout=15)
            print(f"DefiLlama {protocol_slug} → Status: {r.status_code}")

            # If it's not JSON (HTML error page) print the first 300 characters for diagnosis
            if not r.text.strip().startswith(("{", "[")):
                print("Non-JSON response received. First 300 chars:\n", r.text[:300])

            r.raise_for_status()
            data = r.json()

            # DefiLlama response often has top-level "tvl" (aggregate) but sometimes only per-chain. This handles both safely.
            if "tvl" in data and isinstance(data["tvl"], list) and len(data["tvl"]) > 0:
                tvl_list = data["tvl"]
            else:
                raise KeyError(f"No 'tvl' array found in response for {protocol_slug}")

            tvl_data = pd.DataFrame(tvl_list)
            tvl_data["date"] = pd.to_datetime(tvl_data["date"], unit="s")
            tvl_data = tvl_data[["date", "totalLiquidityUSD"]].rename(columns={"totalLiquidityUSD": "tvl"})
            tvl_data = tvl_data.dropna()

            last_date = tvl_data["date"].iloc[-1] if not tvl_data.empty else None

            return tvl_data, last_date, data

        except Exception as e:
            print(f"DefiLlama attempt {attempt+1} for {protocol_slug} failed: {e}")
            time.sleep(2 ** attempt)

    raise Exception(f"Failed to fetch TVL for {protocol_slug} after retries")

In [ ]:
#  METRIC CALCULATIONS

def calculate_tvl_growth(tvl_df):
    tvl_df = tvl_df.dropna().sort_values("date")
    if len(tvl_df) < 30:
        return 0.0

    end_date = tvl_df["date"].max()
    start_date = end_date - pd.Timedelta(days=365)
    recent = tvl_df[tvl_df["date"] >= start_date]

    days = (recent["date"].iloc[-1] - recent["date"].iloc[0]).days
    if days <= 0:
        return 0.0

    start_tvl = recent["tvl"].iloc[0]
    end_tvl = recent["tvl"].iloc[-1]
    if start_tvl <= 0:
        return 0.0

    cagr = (end_tvl / start_tvl) ** (365 / days) - 1
    return cagr



def calculate_future_dilution_potential(circ, total, max_s):
    cap = max_s if max_s is not None else total
    if cap is None or cap == 0:
        return 0.0
    return max(0.0, (cap - circ) / cap)



def emission_pressure(supply_overhang, tvl_growth):
    """
    Heuristic metric combining supply overhang (undistributed supply ratio) with TVL momentum.
    Not derivable from standard financial theory - designed as a relative ranking tool within a peer group.
    Rationale: Positive TVL growth can absorb unlock pressure (pressure = overhang / growth).
    Negative growth amplifies pressure with a penalty (doubles overhang + scales with decline severity).
    Both branches capped at 10. Produces a comparable spread across protocols but should not be
    interpreted as a precise measure of annual dilution rate.
    """

    if tvl_growth > 0:
        raw = supply_overhang / max(tvl_growth, 0.05)       # Floor at 5% growth to prevent near-zero TVL gains creating extreme pressure scores
        return min(raw, 10.0)                               # Cap both branches at 10 for consistency
    else:
        penalty = abs(tvl_growth) * 2.0                     # Scales with how negative (e.g., -50% = +1.0 extra)
        return min(supply_overhang * 2.0 + penalty, 10.0)   # Cap at 10 to avoid extremes




# MAIN ANALYSIS

results = {}

for name, config in PROTOCOLS.items():
    print(f"\n  Processing {name}  ")
    try:
        cg = fetch_coingecko_data(config["coingecko_id"], API_KEY)
        tvl_df, last_tvl_date, full_data = fetch_tvl_history(config["defillama_slug"])
        tvl_growth = calculate_tvl_growth(tvl_df)
        supply_overhang = calculate_future_dilution_potential(cg["circulating_supply"], cg["total_supply"], cg["max_supply"])

        # Fetch and annualize revenue using the correct DefiLlama endpoint (handles all protocols)
        annual_estimated_revenue = np.nan
        revenue_url = f"https://api.llama.fi/summary/fees/{config['defillama_slug']}?dataType=dailyRevenue"
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36"
        }
        try:
            r = requests.get(revenue_url, headers=headers, timeout=15)
            r.raise_for_status()
            revenue_data = r.json()
            chart = revenue_data.get("totalDataChart", [])
            if chart:
                revenue_df = pd.DataFrame(chart, columns=["date", "dailyRevenue"])
                revenue_df["date"] = pd.to_datetime(revenue_df["date"], unit="s")
                recent_revenue = revenue_df.tail(30)
                if not recent_revenue.empty:
                    daily_avg_revenue = recent_revenue["dailyRevenue"].mean()
                    annual_estimated_revenue = daily_avg_revenue * 365
        except Exception as e:
            print(f"Revenue fetch failed for {name}: {e}")

        if not np.isnan(annual_estimated_revenue) and annual_estimated_revenue != 0:
            ps_ratio = cg["market_cap"] / annual_estimated_revenue
            supply_for_fdv = cg["max_supply"] or cg["total_supply"]
            fdv = cg["price"] * supply_for_fdv if supply_for_fdv else cg["market_cap"]
            fdv_ps = fdv / annual_estimated_revenue
            print(f"   Annual Revenue (30d avg):  ${annual_estimated_revenue:,.0f}")
        else:
            ps_ratio = np.nan
            fdv_ps = np.nan
            print(f"   Annual Revenue (30d avg):  N/A")



        base_metrics = {
            "market_cap": cg["market_cap"],
            "tvl": tvl_df["tvl"].iloc[-1],
            "supply_overhang": supply_overhang,
            "tvl_growth": tvl_growth,
            "emission_pressure": emission_pressure(supply_overhang, tvl_growth),
            "ps_ratio": ps_ratio,
            "fdv_ps": fdv_ps,
            "last_tvl_date": last_tvl_date.strftime("%Y-%m-%d") if last_tvl_date else "Unknown"
        }
        results[name] = {"base": base_metrics}

        print(f"   TVL Growth (annualized):   {tvl_growth:+.2%}")
        print(f"   Future Dilution Potential: {supply_overhang:.4f}  (undistributed supply / max supply)")
        print(f"   Emission Pressure:         {base_metrics['emission_pressure']:.2f}")
        print(f"   P/S Ratio:                 {base_metrics['ps_ratio']:.1f}")
        print(f"   FDV/Revenue:               {base_metrics['fdv_ps']:.1f}")

    except requests.exceptions.HTTPError as e:
        print(f"Error fetching data for {name}: {e}")
        print(f"Response content: {e.response.text}")
    except Exception as e:
        print(f"An unexpected error occurred for {name}: {e}")
    finally:
        time.sleep(3)




# NORMALIZATION FOR RADAR CHART

def min_max_normalize(values):
    min_val = min(values)
    max_val = max(values)
    if max_val - min_val == 0:
        return [0.5 for _ in values]
    return [(v - min_val) / (max_val - min_val) for v in values]

protocol_names = list(results.keys())

metrics = {
    "Future Dilution Potential": [results[p]["base"]["supply_overhang"] for p in protocol_names],
    "Emission Pressure": [results[p]["base"]["emission_pressure"] for p in protocol_names],
    "Valuation Risk (P/S)": [results[p]["base"].get("ps_ratio", np.nan) for p in protocol_names],
    "TVL Instability": [max(0, -results[p]["base"]["tvl_growth"]) for p in protocol_names],
    "FDV / Revenue": [results[p]["base"].get("fdv_ps", np.nan) for p in protocol_names]
}

# Apply log scaling to outlier-prone metrics for better normalization without hard caps
for key in ["Valuation Risk (P/S)", "FDV / Revenue", "Emission Pressure"]:
    metrics[key] = [np.log1p(max(v, 0)) if not np.isnan(v) else np.nan for v in metrics[key]]  # log1p for positive values, handles 0

normalized_metrics = {}
for metric_name, values in metrics.items():
    normalized_metrics[metric_name] = min_max_normalize(values)




# FINAL COMPOSITE SCORE USING NORMALIZED METRICS (0-1 scale)

for idx, p in enumerate(protocol_names):
    base = results[p]["base"]
    has_revenue = not np.isnan(base.get("ps_ratio", np.nan))
    accrues = PROTOCOLS[p].get("revenue_accrues_to_holder", True)

    if has_revenue and accrues:
        # Full 5-metric weighting: P/S is a genuine holder value signal
        base["composite_score"] = (
            0.25 * normalized_metrics["Future Dilution Potential"][idx] +
            0.20 * normalized_metrics["Emission Pressure"][idx] +
            0.20 * normalized_metrics["Valuation Risk (P/S)"][idx] +
            0.20 * normalized_metrics["TVL Instability"][idx] +
            0.15 * normalized_metrics["FDV / Revenue"][idx]
        )
    elif has_revenue and not accrues:
        # P/S exists at protocol level but doesn't flow to token holders (e.g. Ethena, Lido)
        # Halve P/S weight and redistribute 5% each to supply overhang and emission pressure where dilution risk is the more relevant holder concern
        base["composite_score"] = (
            0.30 * normalized_metrics["Future Dilution Potential"][idx] +
            0.25 * normalized_metrics["Emission Pressure"][idx] +
            0.10 * normalized_metrics["Valuation Risk (P/S)"][idx] +
            0.20 * normalized_metrics["TVL Instability"][idx] +
            0.15 * normalized_metrics["FDV / Revenue"][idx]
        )
    else:
        # No revenue data: redistribute weight across 3 available metrics
        base["composite_score"] = (
            0.40 * normalized_metrics["Future Dilution Potential"][idx] +
            0.35 * normalized_metrics["Emission Pressure"][idx] +
            0.25 * normalized_metrics["TVL Instability"][idx]
        )



# Radar chart function
colors = [COLORS[p] for p in protocol_names]

def plot_radar(protocol_index, color):
    labels = list(normalized_metrics.keys())
    values = [normalized_metrics[label][protocol_index] for label in labels]
    values += values[:1]

    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    ax.plot(angles, values, linewidth=2.5, color=color, label=protocol_names[protocol_index])
    ax.fill(angles, values, alpha=0.25, color=color)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_title(f"{protocol_names[protocol_index]} Risk Profile\n(Larger Area = Higher Risk)", pad=20, fontsize=13)
    ax.set_ylim(0, 1)
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

for i in range(len(protocol_names)):
    plot_radar(i, colors[i])
    plt.savefig(f"{OUTPUT_FOLDER}/{protocol_names[i]}_radar.png", bbox_inches='tight', dpi=200, facecolor='white')
    plt.close() # Close the figure to free memory and prevent conflicts




# VISUALIZATION

inflations = [results[p]["base"]["supply_overhang"] for p in protocol_names]
pressures = [results[p]["base"]["emission_pressure"] for p in protocol_names]


def dilution_projection(circ, total, max_s, years=3):
    cap = max_s if max_s is not None else total
    if cap is None or cap == 0:
        return [circ] * (years + 1)          # uncapped flat line

    remaining = max(0.0, cap - circ)
    if remaining == 0:
        return [circ] * (years + 1)

    annual_unlock = remaining / years
    projections = []
    current = circ
    for _ in range(years + 1):
        projections.append(current)
        current += annual_unlock
    return projections

years = list(range(0, 4))


# Fetch projections once, reuse for both panels
supply_data = {}
for name, config in PROTOCOLS.items():
    cg = fetch_coingecko_data(config["coingecko_id"], API_KEY)
    projection = dilution_projection(cg["circulating_supply"], cg["total_supply"], cg["max_supply"])
    supply_data[name] = [p / 1e6 for p in projection]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for name, proj in supply_data.items():
    ax1.plot(years, proj, label=name, marker='o', linewidth=2, color=COLORS[name])
    ax2.plot(years, proj, label=name, marker='o', linewidth=2, color=COLORS[name])

ax1.set_title("Linear Scale")
ax1.set_xlabel("Years from now")
ax1.set_ylabel("Projected Circulating Supply (millions)")
ax1.set_xticks([0, 1, 2, 3])
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_yscale("log")
ax2.set_title("Log Scale (separates smaller protocols)")
ax2.set_xlabel("Years from now")
ax2.set_ylabel("Projected Circulating Supply (millions, log scale)")
ax2.set_xticks([0, 1, 2, 3])
ax2.legend()
ax2.grid(True, alpha=0.3, which="both")

fig.suptitle("3-Year Circulating Supply Projection (in millions)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_FOLDER}/supply_projection_millions.png", bbox_inches='tight', dpi=200, facecolor='white')
plt.close()




# RELATIVE DILUTION % CHART

plt.figure(figsize=(10, 6))

for name, config in PROTOCOLS.items():
    cg = fetch_coingecko_data(config["coingecko_id"], API_KEY)
    projection = dilution_projection(
        cg["circulating_supply"],
        cg["total_supply"],
        cg["max_supply"])
    if projection:
        years = list(range(0, 4))
        # Normalize to % of max supply
        max_cap = cg["max_supply"] if cg["max_supply"] is not None else cg["total_supply"]
        if max_cap and max_cap > 0:
            rel = [ (p / max_cap) * 100 for p in projection ]
            plt.plot(years, rel, label=name, marker='o', linewidth=2.5, color=COLORS[name])

plt.title("Relative Dilution - % of Max Supply Unlocked (3-Year Projection)")
plt.xlabel("Years from now")
plt.ylabel("Circulating Supply (% of Max Cap)")
plt.xticks([0, 1, 2, 3])
plt.ylim(0, 105)
plt.grid(True, alpha=0.3)
plt.legend()
plt.savefig(f"{OUTPUT_FOLDER}/relative_dilution.png", bbox_inches='tight', dpi=200, facecolor='white')
plt.close()




# FINAL COMPOSITE SCORE RANKING

plt.figure(figsize=(8, 5))

# Sort worst-to-best (descending order) so the chart reads as a clear ranking left to right
sorted_protocols = sorted(protocol_names, key=lambda p: results[p]["base"]["composite_score"], reverse=True)
scores = [results[p]["base"]["composite_score"] for p in sorted_protocols]
colors_bar = [COLORS[p] for p in sorted_protocols]

plt.bar(sorted_protocols, scores, color=colors_bar)
plt.title("Sustainability Composite Score (LOWER = Better)", fontsize=14)
plt.ylabel("Composite Risk Score")
plt.grid(axis='y', alpha=0.3)

for i, v in enumerate(scores):
    plt.text(i, v + 0.01, f"{v:.3f}", ha='center', fontsize=11)

plt.savefig(f"{OUTPUT_FOLDER}/composite_score_ranking.png", bbox_inches='tight', dpi=200, facecolor='white')
plt.close()

print(f"\nTop Performer: {sorted_protocols[scores.index(min(scores))]} exhibits the lowest composite risk score.")


In [ ]:
#   DYNAMIC TEXTS FOR PDF (makes everything auto-update)
protocol_names = list(PROTOCOLS.keys())

# Winner
winner = min(protocol_names, key=lambda p: results[p]["base"]["composite_score"])
win_score = results[winner]["base"]["composite_score"]

# TVL in billions for descriptions
tvl_b = {p: results[p]["base"]["tvl"] / 1_000_000_000 for p in protocol_names}

# 3-year dilution %
def get_dilution_pct(name):
    cg = fetch_coingecko_data(PROTOCOLS[name]["coingecko_id"], API_KEY)
    proj = dilution_projection(cg["circulating_supply"], cg["total_supply"], cg["max_supply"], years=3)
    start = proj[0]
    end = proj[-1]
    if start > 0:
        return round(((end - start) / start) * 100)
    return 0

dil = {name: get_dilution_pct(name) for name in protocol_names}



# Dynamic strings
dynamic_summary = (
    "This report delivers a concise, data-driven assessment of long-term token sustainability "
    "for six core DeFi protocols using live CoinGecko and DefiLlama data. We calculate five "
    "critical metrics - annualized TVL growth, future dilution potential from supply schedules, emission "
    "pressure relative to growth, P/S ratio based on real fee revenue, and FDV/Revenue - then "
    "combine them into a single composite risk score (lower = more sustainable). "
    "Emission Pressure is a heuristic metric: when TVL growth is positive, it equals "
    "supply_overhang / TVL_growth (floored at 5% growth); when negative, it equals "
    "(supply_overhang x 2) + (|decline| x 2). Both branches are capped at 10 to avoid extremes. "
    "It measures how much undistributed supply pressure is absorbed or amplified by protocol momentum. "
    "Standard composite weights (protocols where revenue accrues to token holders): "
    "Future Dilution Potential 25%, Emission Pressure 20%, Valuation Risk (P/S) 20%, "
    "TVL Instability 20%, FDV/Revenue 15%. "
    "Adjusted weights (governance-only tokens where protocol revenue does not flow to holders, e.g. Ethena, Lido): "
    "Future Dilution Potential 30%, Emission Pressure 25%, Valuation Risk (P/S) 10%, "
    "TVL Instability 20%, FDV/Revenue 15%. "
    "Note: Composite scores are relative to this peer group; values will shift with different protocol selections."
)

pendle_text = (
    f"Pendle (PENDLE) - the leading yield-tokenization protocol with ${tvl_b['Pendle']:.1f}B TVL - "
    f"was chosen because it pioneered PT/YT splitting for fixed and leveraged yield trading. "
    f"High emissions for liquidity incentives ({results['Pendle']['base']['supply_overhang']:.4f} supply overhang ratio) but strong revenue capture "
    f"make it the perfect example of 2026 DeFi primitives. Note: TVL peaked near $7B in mid-2024 "
    f"driven by Eigenlayer and Ethena points-farming programs; the current figure reflects organic "
    f"demand after those incentive cycles concluded."
)

ethena_text = (
    f"Ethena (ENA) - the pioneer of synthetic dollar (USDe) and delta-neutral yield with ${tvl_b['Ethena']:.1f}B+ TVL - "
    f"was included as the breakout stablecoin innovation of 2025-2026. With {results['Ethena']['base']['supply_overhang']:.4f} supply overhang ratio "
    f"and {results['Ethena']['base']['tvl_growth']:+.2%} TVL growth, it shows funding-rate mechanics creating sustainable yield."
)

gmx_text = (
    f"GMX (GMX) - the benchmark on-chain perpetuals DEX with fee-sharing to stakers (note: V1 distributed 30% to GMX stakers; V2 revised the split across GLP/GM pools) - "
    f"was selected to represent real revenue-aligned tokenomics. With controlled inflation and proven holder value accrual, "
    f"it contrasts emission-heavy models despite {results['GMX']['base']['tvl_growth']:+.2%} TVL growth."
)

lido_text = (
    f"Lido (LDO) - the dominant liquid staking protocol with ${tvl_b['Lido']:.1f}B+ TVL - "
    f"was chosen as the leading example of staking innovation. With {results['Lido']['base']['supply_overhang']:.4f} supply overhang ratio "
    f"and {results['Lido']['base']['tvl_growth']:+.2%} TVL growth, it shows how real utility drives sustainable token value."
)

curve_text = (
    f"Curve Finance (CRV) - the premier stablecoin DEX with its iconic veCRV locking model - highlights advanced "
    f"tokenomics in competitive markets. Its {results['Curve Finance']['base']['supply_overhang']:.4f} supply overhang ratio and "
    f"{results['Curve Finance']['base']['tvl_growth']:+.2%} growth illustrate the trade-offs of emission-heavy designs."
)

sky_text = (
    f"Sky (SKY, formerly MakerDAO) - the original decentralized stablecoin issuer - represents mature "
    f"tokenomics with buybacks and collateral interest. Despite {results['Sky']['base']['tvl_growth']:+.2%} TVL growth "
    f"and {results['Sky']['base']['supply_overhang']:.4f} future dilution potential, it remains a benchmark for long-term holder alignment. "
    f"Note: supply metrics reflect the post-rebrand state (1 MKR = 24,000 SKY conversion), which substantially inflates "
    f"circulating supply figures and should be interpreted in that context."
)

charts_text = (
    f"- Relative Dilution chart shows exactly what percentage of each token's maximum supply will unlock "
    f"over the next 3 years - revealing future selling pressure on holders. Note: Assumes linear unlocks for simplicity; actual vesting may be non-linear or emissions-based.\n"
    f"- 3-Year Circulating Supply Projection displays absolute token growth in millions, allowing direct "
    f"visual comparison of emission scale across protocols.\n"
    f"- Sustainability Composite Score Ranking bar chart synthesizes all five metrics into one final score. "
    f"{winner} exhibits the lowest score at {win_score:.3f}, indicating the strongest sustainability among the six."
)



# Sort protocols by 3-year dilution % so the LLM always gets correctly ranked context
dil_sorted = sorted(dil.items(), key=lambda x: x[1])
dil_lowest  = dil_sorted[:2]        # 2 best (least dilution)
dil_highest = dil_sorted[-2:]       # 2 worst (most dilution)
dil_middle  = dil_sorted[2:-2]      # remainder

relative_dilution_text = (
    "3-year supply increase by protocol (% of circulating supply at start): "
    + ", ".join([f"{p} {v}%" for p, v in dil_sorted])
    + ". LOWER % = less dilution pressure on holders."
)

supply_projection_text = (
    f"Lowest dilution: {', '.join([f'{p} ({v}%)' for p, v in dil_lowest])}. "
    f"Mid-range: {', '.join([f'{p} ({v}%)' for p, v in dil_middle])}. "
    f"Highest dilution: {', '.join([f'{p} ({v}%)' for p, v in dil_highest])}."
)

composite_summary_metrics = " | ".join([f"{p}: {results[p]['base']['composite_score']:.3f}" for p in protocol_names])

In [ ]:
# PDF GENERATION CODE

pdf = FPDF(orientation='P', unit='mm', format='A4')
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()



# Title Page

# Main Title
pdf.set_font("Arial", "B", 18)
pdf.cell(0, 15, txt="DeFi Token Sustainability Report", ln=True, align='C')
pdf.ln(8)



# Executive Summary
pdf.set_font("Arial", "B", 13)
pdf.cell(0, 8, txt="Executive Summary", ln=True, align='C')
pdf.ln(4)

pdf.set_font("Arial", "", 10.8)
summary = dynamic_summary
pdf.multi_cell(0, 5.2, txt=summary)
pdf.ln(7)



# Protocols Analyzed
pdf.set_font("Arial", "B", 12)
pdf.cell(0, 7, txt="Protocols Analyzed & Why They Were Chosen", ln=True)
pdf.ln(3)

pdf.set_font("Arial", "", 10.5)

pdf.multi_cell(0, 5.1, txt=pendle_text)
pdf.ln(3)
pdf.multi_cell(0, 5.1, txt=ethena_text)
pdf.ln(3)
pdf.multi_cell(0, 5.1, txt=gmx_text)
pdf.ln(3)
pdf.multi_cell(0, 5.1, txt=lido_text)
pdf.ln(3)
pdf.multi_cell(0, 5.1, txt=curve_text)
pdf.ln(3)
pdf.multi_cell(0, 5.1, txt=sky_text)
pdf.ln(7)



# Visual Insights
pdf.set_font("Arial", "B", 12)
pdf.cell(0, 7, txt="What the Final Three Charts Reveal", ln=True)
pdf.ln(3)

pdf.set_font("Arial", "", 10.5)
pdf.multi_cell(0, 5.2, txt=charts_text)
pdf.ln(5)

pdf.set_font("Arial", "I", 9.5)
pdf.cell(0, 5, txt=f"All data pulled {datetime.now().strftime('%B %Y')} - One-click Colab - Ready for investor decks", ln=True, align='C')



# Metric Gloassary Page

pdf.add_page()

pdf.set_font("Arial", "B", 14)
pdf.cell(0, 10, txt="Metric Definitions", ln=True, align='C')
pdf.ln(2)
pdf.set_font("Arial", "I", 9)
pdf.cell(0, 5, txt="Reference guide to every metric shown in the protocol tables.", ln=True, align='C')
pdf.ln(7)

glossary_entries = [
    (
        "Market Cap",
        "Total market value of all tokens currently in circulation (price x circulating supply). "
        "Low: smaller protocol by market valuation. High: larger, more established market presence."
    ),
    (
        "TVL (Total Value Locked)",
        "Total assets actively deposited and utilized within the protocol. Reflects real capital trust and usage. "
        "Low: limited capital engagement or early-stage protocol. High: strong user adoption and capital depth."
    ),
    (
        "TVL Growth (ann.)",
        "Annualized CAGR of TVL over the trailing 12 months. Measures whether the protocol is growing or shrinking. "
        "Negative: capital or users are exiting. Positive: growing adoption and inflows."
    ),
    (
        "TVL Instability",
        "Derived from TVL Growth: max(0, -TVL_growth). Protocols with positive TVL growth score 0 on this dimension "
        "(no instability penalty), while declining protocols are penalized proportionally to their rate of contraction. "
        "This means a fast-growing protocol like Ethena receives a perfect 0 here despite other risk factors."
    ),
    (
        "Future Dilution Potential",
        "Proportion of the maximum token supply not yet in circulation: (max_supply - circ_supply) / max_supply. Scale: 0 to 1. "
        "Low (near 0): most supply is already circulating - minimal future unlock pressure on holders. "
        "High (near 1): large proportion of supply still to be released - greater future sell pressure risk. "
        "Note: assumes all remaining supply will eventually circulate. For protocols with continuous emission "
        "schedules (e.g. CRV), actual release rate may differ substantially from what this figure implies."
    ),
    (
        "Emission Pressure",
        "Heuristic metric combining supply overhang (undistributed supply as a fraction of max supply) "
        "with TVL momentum as a relative ranking signal. Not derived from standard financial theory. "
        "Positive TVL growth: overhang / TVL_growth (floored at 5% to prevent near-zero growth from "
        "creating extreme scores). Negative TVL growth: (overhang x 2) + (|TVL decline| x 2). "
        "Both branches capped at 10. Low: unlock pressure is well-absorbed by growth momentum. "
        "High: significant supply overhang against weak or declining protocol traction. "
        "Scores are only meaningful relative to this peer group."
    ),
    (
        "P/S Ratio (Price-to-Sales)",
        "Market Cap divided by annualized protocol revenue (30-day average daily revenue x 365). "
        "Low: token is inexpensive relative to current revenue generation - potentially undervalued. "
        "High: token trades at a revenue premium - market expects significant future growth. "
        "Governance-only tokens (e.g. ENA, LDO) where revenue flows to the protocol treasury rather "
        "than token holders are assigned half the standard P/S weight in the composite score, as "
        "market cap / protocol revenue is not a direct holder value metric in those cases."
    ),
    (
        "FDV / Revenue",
        "Fully Diluted Valuation (price x max supply) divided by annualized revenue. A forward-looking valuation check. "
        "Low: reasonable forward valuation relative to current earnings. "
        "High: protocol is priced for aggressive future revenue growth well beyond current levels."
    ),
    (
        "Composite Score",
        "Weighted sum of all five risk metrics, each min-max normalized within this peer group (0-1 scale). "
        "Standard weights (protocols where revenue accrues to token holders): "
        "Future Dilution Potential 25% | Emission Pressure 20% | Valuation Risk (P/S) 20% | "
        "TVL Instability 20% | FDV/Revenue 15%. "
        "Adjusted weights (protocols where protocol revenue does not flow to token holders, e.g. Ethena, Lido): "
        "Future Dilution Potential 30% | Emission Pressure 25% | Valuation Risk (P/S) 10% | "
        "TVL Instability 20% | FDV/Revenue 15%. The P/S weight is halved for governance-only tokens "
        "because market cap / protocol revenue is not a holder value metric in those cases. "
        "Implementation note: P/S, FDV/Revenue, and Emission Pressure are log-scaled (log1p) before "
        "min-max normalization to reduce distortion from extreme outliers. "
        "LOWER score = more sustainable tokenomics. Scores are relative to this peer group and will shift "
        "if the protocol selection changes."
    ),
]

for metric_name, explanation in glossary_entries:
    pdf.set_font("Arial", "B", 9.5)
    pdf.cell(0, 6, txt=metric_name, ln=True)
    pdf.set_font("Arial", "", 9)
    pdf.multi_cell(0, 4.8, txt=explanation)
    pdf.ln(3.5)

In [ ]:
# LLM COMMENTARY FUNCTION

def get_llm_commentary(title: str, metrics: str, context: str = '') -> str:

    prompt = f"""
    You are a concise, professional DeFi token analyst writing for institutional investors.

    CRITICAL DEFINITION - Emission Pressure measures token inflation relative to TVL growth momentum,
    NOT relative to protocol revenue. Never describe it as exceeding or comparing to revenues.
    If Emission Pressure > 1, describe it as: emissions outpacing TVL growth absorption capacity.

    Write EXACTLY three sentences (maximum 110 words total) about {title}'s long-term token sustainability.

    Rules:
    - Sentence 1: One positive or neutral observation using the strongest metric (TVL, inflation, or valuation).
    - Sentence 2: One risk observation using the weakest metric (TVL growth, emission pressure, or valuation).
    - Sentence 3: Conclude with the Composite Score implication. Always use the exact phrase "Composite Score of X.XXX". IMPORTANT: a LOWER composite score means HIGHER sustainability. State clearly whether this score places the protocol among the most sustainable (lowest scores) or least sustainable (highest scores) in the peer group.

    Use ONLY these exact metrics:
    {metrics}

    Context: {context}

    Tone: strictly neutral, data-driven, no hype words (excellent, strong, superior, robust, compelling, impressive, etc.).
    Always start directly with analysis. No introductions.
    """

    try:
        response = ai.generate_text(prompt)
        return sanitize(response.strip())
    except Exception as e:
        return "AI commentary temporarily unavailable (Colab AI service issue)."




def get_chart_commentary(chart_context: str, insight_angle: str = '') -> str:
    prompt = f"""
    You are a professional DeFi analyst writing chart commentary for an institutional investor report.

    Write a structured paragraph of EXACTLY four sentences (100-140 words total) interpreting this cross-protocol data:
    {chart_context}

    Angle: {insight_angle}

    Rules:
    - Sentence 1: State why this metric matters to token holders and what the chart is designed to reveal.
    - Sentence 2: Name the standout protocol(s) at the top and bottom of the range with their exact figures.
    - Sentence 3: Explain the practical investor implication of that spread - what it means for buying, holding, or selling decisions.
    - Sentence 4: Identify which protocol(s) sit in the middle of the range and briefly characterize their position relative to the extremes.
    - Do NOT assign or mention any Composite Score - this is a chart-level observation only.
    - Do NOT use hype words (excellent, impressive, robust, compelling, superior).
    - Tone: strictly neutral, data-driven, institutional. No introductions or preamble.
    """
    try:
        response = ai.generate_text(prompt)
        return sanitize(response.strip())
    except Exception as e:
        return "Chart commentary temporarily unavailable."




# PROTOCOL PAGES

for i, protocol in enumerate(protocol_names):
    pdf.add_page()
    base = results[protocol]["base"]

    # Protocol name
    pdf.set_font("Arial", "B", 14)
    pdf.cell(0, 12, txt=protocol, ln=True, align='C')
    pdf.ln(8)

    # Centered Table
    pdf.set_font("Arial", "B", 10)
    table_width = 125
    pdf.set_x((210 - table_width) / 2)

    pdf.cell(80, 8, "Metric", 1, align='C')
    pdf.cell(45, 8, "Value", 1, align='C')
    pdf.ln()

    pdf.set_font("Arial", "", 9.5)
    table_data = [
        ("Market Cap",                f"${base['market_cap']:,}"),
        ("TVL",                       f"${int(round(base['tvl'])):,}"),
        ("TVL as of",                 base.get("last_tvl_date", "Unknown")),
        ("Future Dilution Potential", f"{base['supply_overhang']:.4f}"),
        ("TVL Growth (ann.)",         f"{base['tvl_growth']:+.2%}"),
        ("Emission Pressure",         f"{base['emission_pressure']:.3f}"),
        ("P/S Ratio",                 "N/A" if np.isnan(base.get('ps_ratio', 0)) else f"{base['ps_ratio']:.2f}"),
        ("FDV / Revenue",             "N/A" if np.isnan(base.get('fdv_ps', 0)) else f"{base['fdv_ps']:.2f}"),
        ("Composite Score",           f"{base['composite_score']:.4f}"),
    ]

    for label, value in table_data:
        pdf.set_x((210 - table_width) / 2)
        pdf.cell(80, 7.5, label, 1)
        pdf.cell(45, 7.5, value, 1, align='R')
        pdf.ln()

    pdf.ln(8)



    # AI COMMENTARY

    pdf.set_font("Arial", "B", 11)
    pdf.cell(0, 8, txt="AI Analyst Insight", ln=True)
    pdf.set_font("Arial", "", 10.2)

    all_scores_sorted = sorted(protocol_names, key=lambda p: results[p]["base"]["composite_score"])
    rank = all_scores_sorted.index(protocol) + 1
    n_total = len(protocol_names)
    rank_label = (
        "most sustainable (rank 1)"        if rank == 1 else
        f"2nd most sustainable (rank {rank} of {n_total})"  if rank == 2 else
        f"middle of peer group (rank {rank} of {n_total})"  if rank <= n_total - 2 else
        f"2nd least sustainable (rank {rank} of {n_total})" if rank == n_total - 1 else
        "least sustainable (last place)"
    )

    metrics_str = (
        f"Market Cap: ${base['market_cap']:,.0f} | "
        f"TVL: ${base['tvl']:,.0f} | "
        f"Future Dilution Potential (0-1 scale; HIGHER = larger undistributed supply overhang = greater dilution risk for holders): {base['supply_overhang']:.4f} | "
        f"TVL Growth: {base['tvl_growth']:+.1%} | "
        f"Emission Pressure: {base['emission_pressure']:.3f} | "
        f"P/S: {'N/A' if np.isnan(base.get('ps_ratio', np.nan)) else f'{base['ps_ratio']:.2f}'} | "
        f"FDV/Revenue: {'N/A' if np.isnan(base.get('fdv_ps', np.nan)) else f'{base['fdv_ps']:.2f}'} | "
        f"Composite Score: {base['composite_score']:.4f} | "
        f"Peer group rank (LOWER composite = MORE sustainable): {rank} of {n_total} - this protocol is the {rank_label}. "
        f"You MUST use this exact rank description in sentence 3, do not reinterpret it."
    )

    protocol_context = {
        "Ethena": (
            "Compare briefly to the other protocols where relevant. "
            "Important: ENA token holders have no direct claim on protocol revenue. "
            "Yield from the delta-neutral USDe position accrues to sUSDe stakers, not ENA holders. "
            "This structural disconnect explains the elevated P/S and FDV/Revenue multiples despite "
            "real yield generation within the protocol."
        ),
        "Lido": (
            "Compare briefly to the other protocols where relevant. "
            "MANDATORY: You MUST include in sentence 1 or 2 that Lido's P/S ratio of "
            f"{base['ps_ratio']:.2f} reflects protocol-level earnings that flow to the DAO "
            "treasury - LDO token holders have no direct revenue claim, unlike GMX stakers "
            "or veCRV lockers. This must be stated explicitly."
        ),
    }.get(protocol, "Compare briefly to the other protocols where relevant.")

    commentary = get_llm_commentary(
        protocol,
        metrics_str,
        context=protocol_context
    )

    pdf.multi_cell(0, 5.2, txt=commentary)
    pdf.ln(6)



    # Radar Chart

    radar_file = f"{OUTPUT_FOLDER}/{protocol}_radar.png"
    if os.path.exists(radar_file):
        chart_width = 145
        pdf.set_x((210 - chart_width) / 2)
        pdf.image(radar_file, w=chart_width)

In [ ]:
# FINAL CHARTS

# Relative Dilution page
pdf.add_page()
if os.path.exists(f"{OUTPUT_FOLDER}/relative_dilution.png"):
    pdf.set_font("Arial", "B", 14)
    pdf.cell(0, 12, txt="Relative Dilution - % of Max Supply (3-Year)", ln=True, align='C')
    pdf.ln(10)
    pdf.set_x((210 - 170) / 2)
    pdf.image(f"{OUTPUT_FOLDER}/relative_dilution.png", w=170)



# Methodology footnote
    pdf.ln(6)
    pdf.set_font("Arial", "I", 8.5)
    pdf.set_text_color(100, 100, 100)
    pdf.multi_cell(0, 4.5, txt=sanitize(
        "Methodology note: Projections assume linear unlocks across 3 years. "
        "This is a simplification - actual schedules vary significantly. "
        "CRV emissions follow an exponential decay curve per the Curve whitepaper, "
        "so its 104% figure is an overestimate of realistic near-term dilution. "
        "Lido's LDO has no formal remaining unlock schedule; its figure reflects "
        "undistributed supply only."
    ))

    pdf.set_text_color(0, 0, 0)
    pdf.ln(4)
    pdf.set_font("Arial", "B", 11)
    pdf.cell(0, 8, txt="AI Analyst Insight", ln=True)
    pdf.set_font("Arial", "", 10.2)
    chart_comment = get_chart_commentary(
        relative_dilution_text,
        insight_angle="Explain the long-term selling pressure implication for token holders."
    )
    pdf.multi_cell(0, 5.2, txt=chart_comment)



# Supply Projection page
pdf.add_page()
if os.path.exists(f"{OUTPUT_FOLDER}/supply_projection_millions.png"):
    pdf.set_font("Arial", "B", 14)
    pdf.cell(0, 12, txt="3-Year Circulating Supply Projection (in millions)", ln=True, align='C')
    pdf.ln(10)
    pdf.set_x((210 - 170) / 2)
    pdf.image(f"{OUTPUT_FOLDER}/supply_projection_millions.png", w=170)



 # AI Insight
    pdf.ln(8)
    pdf.set_font("Arial", "B", 11)
    pdf.cell(0, 8, txt="AI Analyst Insight", ln=True)
    pdf.set_font("Arial", "", 10.2)
    chart_comment = get_chart_commentary(
        supply_projection_text,
        insight_angle="Explain how these emission rates affect token price sustainability over 3 years."
    )
    pdf.multi_cell(0, 5.2, txt=chart_comment)



# Composite Score Ranking page
pdf.add_page()
if os.path.exists(f"{OUTPUT_FOLDER}/composite_score_ranking.png"):
    pdf.set_font("Arial", "B", 14)
    pdf.cell(0, 12, txt="Sustainability Composite Score Ranking", ln=True, align='C')
    pdf.ln(15)
    pdf.set_x((210 - 170) / 2)
    pdf.image(f"{OUTPUT_FOLDER}/composite_score_ranking.png", w=170)



 # AI Insight
    pdf.ln(8)
    pdf.set_font("Arial", "B", 11)
    pdf.cell(0, 8, txt="AI Analyst Insight", ln=True)
    pdf.set_font("Arial", "", 10.2)
    chart_comment = get_chart_commentary(
        composite_summary_metrics,
        insight_angle="Explain what separates the top and bottom performers and what this means for investors. LOWER score = MORE sustainable."
    )
    pdf.multi_cell(0, 5.2, txt=chart_comment)

pdf.output("defi_sustainability_report.pdf")
print("Final clean PDF generated!")